Use `evtest` to obtain "Joy-Con (R)":
---

---

In [ ]:
# 🔧 CHANGE THIS to your actual device
JOYCON_DEVICE = "/dev/input/event19"

Key Mapping

----

In [ ]:
from evdev import InputDevice, categorize, ecodes
from evdev import UInput


In [ ]:
# VERTICAL SINGLE HANDED OPERATION
BUTTON_MAP = {
    ecodes.BTN_TR: ecodes.KEY_X,
    ecodes.BTN_TR2: ecodes.KEY_Z,
    ecodes.BTN_NORTH: ecodes.KEY_UP,
    ecodes.BTN_SOUTH: ecodes.KEY_DOWN,
    ecodes.BTN_WEST: ecodes.KEY_LEFT,
    ecodes.BTN_EAST: ecodes.KEY_RIGHT
}


In [ ]:

async def main():
    dev = InputDevice(JOYCON_DEVICE)
    print(f"Listening on {dev.path}: {dev.name}")

    # Create virtual keyboard
    ui = UInput()

    async for event in dev.async_read_loop():
        if event.type == ecodes.EV_KEY:
            key_event = categorize(event)

            if key_event.scancode in BUTTON_MAP:
                mapped_key = BUTTON_MAP[key_event.scancode]

                # Press
                if key_event.keystate == key_event.key_down:
                    ui.write(ecodes.EV_KEY, mapped_key, 1)
                    ui.syn()

                # Release
                elif key_event.keystate == key_event.key_up:
                    ui.write(ecodes.EV_KEY, mapped_key, 0)
                    ui.syn()

await main()

Listening on /dev/input/event19: Joy-Con (R)


In [ ]:
from evdev import InputDevice, categorize, ecodes, UInput
import asyncio

# JOYCON_DEVICE = "/dev/input/event23"

# Axis mappings
AXIS_MAP = {
    ecodes.ABS_RX: ("LEFT", "RIGHT"),
    ecodes.ABS_RY: ("UP", "DOWN"),
}

KEY_LOOKUP = {
    "UP": ecodes.KEY_UP,
    "DOWN": ecodes.KEY_DOWN,
    "LEFT": ecodes.KEY_LEFT,
    "RIGHT": ecodes.KEY_RIGHT,
}

# Tuning
DEADZONE = 20
MAX_VAL = 255  # adjust if your device differs

# Track current pressed state
axis_state = {
    "UP": False,
    "DOWN": False,
    "LEFT": False,
    "RIGHT": False,
}

def handle_axis(ui, axis, value):
    neg_dir, pos_dir = AXIS_MAP[axis]

    # Normalize around center (~128 for Joycon)
    center = MAX_VAL // 2
    delta = value - center

    # Determine direction
    if delta < -DEADZONE:
        active = neg_dir
    elif delta > DEADZONE:
        active = pos_dir
    else:
        active = None

    # Update states
    for direction in [neg_dir, pos_dir]:
        key = KEY_LOOKUP[direction]

        if direction == active:
            if not axis_state[direction]:
                ui.write(ecodes.EV_KEY, key, 1)
                axis_state[direction] = True
        else:
            if axis_state[direction]:
                ui.write(ecodes.EV_KEY, key, 0)
                axis_state[direction] = False

    ui.syn()


async def main():
    dev = InputDevice(JOYCON_DEVICE)
    ui = UInput()

    print(f"Listening on {dev.path}: {dev.name}")

    async for event in dev.async_read_loop():
        if event.type == ecodes.EV_ABS:
            if event.code in AXIS_MAP:
                handle_axis(ui, event.code, event.value)


await main()

Listening on /dev/input/event19: Joy-Con (R)
